# Load Oxford Government AI Readiness Index 2025 → Delta

Reads the official Oxford Insights data workbook (`2025-Government-AI-Readiness-Index-data-1.xlsx`, committed in the repo) and writes two country-level tables to `fso_market_intelligence.frontier_labs`:
- `oxford_ai_readiness` — one row per country: overall score, rank, and the 6 pillars
- `oxford_readiness_dimensions` — long: country × dimension (the 14 dimensions)

Country-level annual index (195 govts). Keyed on `iso3` so it joins with other country data. Overwrite each run; re-run yearly with the new workbook.

In [ ]:
%pip install openpyxl pycountry -q

In [ ]:
CATALOG, SCHEMA, YEAR = "fso_market_intelligence", "frontier_labs", 2025
FILENAME = "2025-Government-AI-Readiness-Index-data-1.xlsx"

import subprocess, os
_hits = subprocess.run(["find", "/Workspace", "-maxdepth", "9", "-name", FILENAME],
                       capture_output=True, text=True).stdout.strip().splitlines()
assert _hits, f"{FILENAME} not found under /Workspace — pull the Git folder first"
XLSX = _hits[0]
print("XLSX =", XLSX)

In [ ]:
import pandas as pd

# 'Dimensions-Pillars' sheet: row 0 = group markers, row 1 = real header → header=1
df = pd.read_excel(XLSX, sheet_name="Dimensions-Pillars", header=1)
df = df[df["Country"].notna()].copy()
df = df.loc[:, ~df.columns.astype(str).str.startswith("Unnamed")]  # drop spacer columns
print("rows:", len(df))
print("columns:", list(df.columns))

In [ ]:
# country -> ISO3 (pycountry fuzzy + overrides for known name mismatches)
import pycountry
OVERRIDES = {
    "United States": "USA", "United States of America": "USA", "Russia": "RUS",
    "South Korea": "KOR", "North Korea": "PRK", "Turkey": "TUR", "T\u00fcrkiye": "TUR",
    "Iran": "IRN", "Syria": "SYR", "Laos": "LAO", "Vietnam": "VNM", "Bolivia": "BOL",
    "Venezuela": "VEN", "Tanzania": "TZA", "Moldova": "MDA", "Brunei": "BRN",
    "Democratic Republic of the Congo": "COD", "DR Congo": "COD", "Congo, Dem. Rep.": "COD",
    "Congo": "COG", "Republic of the Congo": "COG", "Congo, Rep.": "COG",
    "Ivory Coast": "CIV", "C\u00f4te d'Ivoire": "CIV", "Cote d'Ivoire": "CIV",
    "Cape Verde": "CPV", "Swaziland": "SWZ", "Eswatini": "SWZ", "The Gambia": "GMB",
    "Gambia": "GMB", "Kosovo": "XKX", "Palestine": "PSE", "State of Palestine": "PSE",
    "Micronesia": "FSM", "Czechia": "CZE", "Czech Republic": "CZE", "Slovakia": "SVK",
    "United Kingdom": "GBR", "UK": "GBR",
}
def to_iso3(name):
    n = str(name).strip()
    if n in OVERRIDES: return OVERRIDES[n]
    try: return pycountry.countries.lookup(n).alpha_3
    except LookupError:
        try: return pycountry.countries.search_fuzzy(n)[0].alpha_3
        except Exception: return None

df["iso3"] = df["Country"].map(to_iso3)
missing = df[df["iso3"].isna()]["Country"].tolist()
print(f"iso3 mapped: {df['iso3'].notna().sum()}/{len(df)}" + (f" | unmapped: {missing}" if missing else ""))

In [ ]:
from pyspark.sql import functions as F

PILLARS = {
    "Policy Capacity": "policy_capacity", "AI Infrastructure": "ai_infrastructure",
    "Governance": "governance", "Public Sector Adoption": "public_sector_adoption",
    "Development & Diffusion": "development_diffusion", "Resilience": "resilience",
}
ID_COLS = ["Ranking", "Country", "Total Score"] + list(PILLARS)
DIM_COLS = [c for c in df.columns if c not in ID_COLS + ["iso3"]]  # the 14 dimensions (dynamic)
print("dimensions:", DIM_COLS)

# ── main table: country + overall + 6 pillars ──
main = df[["Country", "iso3", "Ranking", "Total Score"] + list(PILLARS)].rename(
    columns={"Country": "country", "Ranking": "rank", "Total Score": "total_score", **PILLARS})
main_sdf = (spark.createDataFrame(main).withColumn("year", F.lit(YEAR))
            .withColumn("rank", F.col("rank").cast("int")).withColumn("captured_at", F.current_date()))
(main_sdf.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.oxford_ai_readiness"))
print("oxford_ai_readiness rows:", main_sdf.count())

# ── long table: country × dimension ──
long = df[["Country", "iso3"] + DIM_COLS].melt(id_vars=["Country", "iso3"], var_name="dimension", value_name="score")
long = long.rename(columns={"Country": "country"})
dim_sdf = (spark.createDataFrame(long).withColumn("year", F.lit(YEAR)).withColumn("captured_at", F.current_date()))
(dim_sdf.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.oxford_readiness_dimensions"))
print("oxford_readiness_dimensions rows:", dim_sdf.count())
display(main_sdf.orderBy("rank").limit(10))